In [ ]:
%pip install -q -U groq

In [ ]:
from datetime import datetime, timezone
from google.colab import drive, userdata
from groq import Groq
from io import BytesIO
from pathlib import Path
from PIL import Image

import base64
import json
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio


# --------------------------------------------------
# Mount Drive and configure paths
# --------------------------------------------------

drive.mount(
    "/content/drive"
)

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "school_imagery_assignment"
)

MASTER_METADATA_PATH = (
    PROJECT_DIRECTORY
    / "naip_master_1200m"
    / "naip_master_metadata.csv"
)

QWEN_OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "qwen_predictions"
)

QWEN_JSON_DIRECTORY = (
    QWEN_OUTPUT_DIRECTORY
    / "raw_json"
)

QWEN_PREDICTIONS_PATH = (
    QWEN_OUTPUT_DIRECTORY
    / "qwen_predictions_raw.csv"
)

QWEN_JSON_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Load imagery metadata
# --------------------------------------------------

master_metadata = pd.read_csv(
    MASTER_METADATA_PATH,
    dtype={
        "school_id": "string",
    },
)

if len(master_metadata) != 25:
    raise ValueError(
        f"Expected 25 schools, "
        f"but found {len(master_metadata)}."
    )

print(
    "Schools loaded:",
    len(master_metadata),
)


# --------------------------------------------------
# Configure Groq
# --------------------------------------------------

GROQ_MODEL_ID = (
    "qwen/qwen3.6-27b"
)

PROMPT_VERSION = (
    "qwen_aerial_attributes_v2_rewrite_all"
)

REQUEST_INTERVAL_SECONDS = 65
MAX_REQUEST_ATTEMPTS = 2

api_key = userdata.get(
    "GROQ_API_KEY"
)

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY was not found in "
        "Colab Secrets."
    )

client = Groq(
    api_key=api_key
)


# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def safe_filename(value):
    safe_value = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value),
    )

    return safe_value.strip("_")


def load_geotiff_image(
    geotiff_path,
):
    with rasterio.open(
        geotiff_path
    ) as source:
        rgb_array = source.read(
            [1, 2, 3]
        )

        resolution_x = abs(
            source.transform.a
        )

        resolution_y = abs(
            source.transform.e
        )

    rgb_array = np.moveaxis(
        rgb_array,
        0,
        -1,
    )

    if rgb_array.dtype != np.uint8:
        valid_pixels = rgb_array[
            np.any(
                rgb_array != 0,
                axis=2,
            )
        ]

        if not valid_pixels.size:
            raise ValueError(
                "GeoTIFF contains no valid RGB pixels."
            )

        lower, upper = np.percentile(
            valid_pixels,
            [2, 98],
        )

        if upper <= lower:
            raise ValueError(
                "Unable to scale RGB values."
            )

        rgb_array = np.clip(
            (
                rgb_array.astype(
                    np.float32
                )
                - lower
            )
            / (upper - lower),
            0,
            1,
        )

        rgb_array = (
            rgb_array * 255
        ).astype(
            np.uint8
        )

    pil_image = Image.fromarray(
        rgb_array
    ).convert("RGB")

    return (
        pil_image,
        resolution_x,
        resolution_y,
    )


def image_to_data_url(
    pil_image,
    quality=85,
):
    buffer = BytesIO()

    pil_image.convert(
        "RGB"
    ).save(
        buffer,
        format="JPEG",
        quality=quality,
        optimize=True,
    )

    encoded_image = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return (
        "data:image/jpeg;base64,"
        + encoded_image
    )


def parse_json_response(
    response_text,
):
    try:
        return json.loads(
            response_text
        )

    except json.JSONDecodeError:
        cleaned_response = (
            response_text
            .strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )

        return json.loads(
            cleaned_response
        )


def make_predictions_table(
    results,
):
    table = pd.DataFrame(
        results
    )

    for json_column in [
        "uncertain_fields",
        "evidence_notes",
    ]:
        if json_column in table.columns:
            table[
                json_column
            ] = table[
                json_column
            ].apply(
                lambda value: (
                    json.dumps(value)
                    if isinstance(
                        value,
                        (dict, list),
                    )
                    else value
                )
            )

    return table


# --------------------------------------------------
# Required model-output fields
# --------------------------------------------------

required_prediction_keys = [
    "campus_visibility",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "uncertain_fields",
    "evidence_notes",
    "human_review_required",
]


# --------------------------------------------------
# Batch prediction
#
# Important:
# There is intentionally no "skip existing file" check.
# Every school is sent to Qwen again.
# --------------------------------------------------

batch_results = []

for position, (_, school) in enumerate(
    master_metadata.iterrows(),
    start=1,
):
    school_id = str(
        school["school_id"]
    )

    school_name = str(
        school["school_name"]
    )

    output_name = (
        f"{school_id}_"
        f"{safe_filename(school_name)}.json"
    )

    json_output_path = (
        QWEN_JSON_DIRECTORY
        / output_name
    )

    image = None
    resolution_x = None
    resolution_y = None
    imagery_vintage = None

    print("\n" + "=" * 70)

    print(
        f"[{position:02d}/"
        f"{len(master_metadata)}] "
        f"{school_name}"
    )

    print("=" * 70)

    try:
        # ------------------------------------------
        # Load this school's aerial image
        # ------------------------------------------

        geotiff_path = Path(
            school["geotiff_file"]
        )

        if not geotiff_path.exists():
            raise FileNotFoundError(
                f"GeoTIFF not found: "
                f"{geotiff_path}"
            )

        (
            image,
            resolution_x,
            resolution_y,
        ) = load_geotiff_image(
            geotiff_path
        )

        if (
            "naip_year" in school.index
            and pd.notna(
                school["naip_year"]
            )
        ):
            imagery_vintage = int(
                school["naip_year"]
            )

        # ------------------------------------------
        # Create two overlapping enlarged strips
        # ------------------------------------------

        strip_height = 1200

        upper_strip = image.crop(
            (
                0,
                0,
                image.width,
                min(
                    strip_height,
                    image.height,
                ),
            )
        )

        lower_start = max(
            0,
            image.height - strip_height,
        )

        lower_strip = image.crop(
            (
                0,
                lower_start,
                image.width,
                image.height,
            )
        )

        overview_data_url = (
            image_to_data_url(
                image
            )
        )

        upper_data_url = (
            image_to_data_url(
                upper_strip
            )
        )

        lower_data_url = (
            image_to_data_url(
                lower_strip
            )
        )

        # ------------------------------------------
        # Define the Qwen prompt
        # ------------------------------------------

        prompt = f"""
You are reviewing NAIP aerial imagery of:

School: {school_name}

The imagery resolution is approximately
{resolution_x} metres per pixel.

Image 1 is the complete 1,200 metre school-area overview.
Image 2 is an enlarged upper portion of Image 1.
Image 3 is an enlarged lower portion of Image 1.

The reported school coordinate may be imperfect. Identify
the likely school campus carefully. If campus identity or
extent is unclear, mark the relevant fields uncertain.

Return exactly one valid JSON object:

{{
  "campus_visibility": "clear, partial, or poor",
  "rooftop_solar_present": "yes, no, or uncertain",
  "rooftop_solar_area_m2_estimate": number or null,
  "portable_classroom_count": integer or null,
  "pool_present": "yes, no, or uncertain",
  "running_track": "yes, no, or uncertain",
  "full_size_sports_fields_count": integer or null,
  "hard_courts_count": integer or null,
  "uncertain_fields": ["field names requiring review"],
  "evidence_notes": {{
    "solar": "brief evidence and approximate location",
    "portable_classrooms": "brief evidence and location",
    "pool": "brief evidence and location",
    "running_track": "brief evidence and location",
    "sports_fields": "brief evidence and location",
    "hard_courts": "brief evidence and location"
  }},
  "human_review_required": true or false
}}

Measurement definitions:

1. Rooftop solar presence:
   Report yes only for recognizable rooftop photovoltaic
   panel arrays. Do not confuse HVAC equipment, skylights,
   roof shadows or dark roofing with solar panels.

2. Rooftop solar area:
   If solar is present, estimate the total visible
   panel-covered area in square metres. This is panel area,
   not total roof area. Otherwise return null.

3. Portable classrooms:
   Count distinct portable or modular classroom buildings.
   Do not count individual rooms. Do not count permanent
   buildings, houses, storage sheds or ordinary trailers.

4. Pool:
   Report yes only for a recognizable swimming pool.
   Do not confuse ponds, blue roofs, artificial turf or
   shadows with pools.

5. Running track:
   Report yes for a recognizable closed oval running track,
   commonly surrounding a rectangular sports field.

6. Full-size sports fields:
   Count recognizable full-size fields intended for sports.
   A field inside a running track counts as one field.
   Do not count small lawns or generic unmarked open spaces.

7. Hard courts:
   Count distinct individual playable outdoor courts with
   recognizable sport markings. Parking lots, parking-space
   markings and paved school yards without court markings
   are not hard courts.

Additional rules:

- Measure only facilities belonging to the target campus.
- Do not generate bounding boxes.
- Use approximate location descriptions such as
  bottom-centre, upper-left or beside the main building.
- Use null or uncertain rather than guessing.
- Do not generate confidence scores.
- These are raw model predictions for later validation
  and human review.
"""

        request_content = [
            {
                "type": "text",
                "text": prompt,
            },
            {
                "type": "text",
                "text": (
                    "Image 1: complete overview"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": overview_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 2: enlarged upper portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": upper_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 3: enlarged lower portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": lower_data_url,
                },
            },
        ]

        # ------------------------------------------
        # Call Groq with retry
        # ------------------------------------------

        candidate_result = None
        last_error = None

        for attempt in range(
            1,
            MAX_REQUEST_ATTEMPTS + 1,
        ):
            try:
                print(
                    f"API attempt "
                    f"{attempt}/"
                    f"{MAX_REQUEST_ATTEMPTS}"
                )

                completion = (
                    client
                    .chat
                    .completions
                    .create(
                        model=GROQ_MODEL_ID,
                        messages=[
                            {
                                "role": "user",
                                "content": (
                                    request_content
                                ),
                            }
                        ],
                        temperature=0.5,
                        reasoning_effort="none",
                        max_completion_tokens=800,
                        response_format={
                            "type": "json_object",
                        },
                    )
                )

                response_text = (
                    completion
                    .choices[0]
                    .message
                    .content
                )

                if not response_text:
                    raise RuntimeError(
                        "Groq returned an empty response."
                    )

                candidate_result = (
                    parse_json_response(
                        response_text
                    )
                )

                missing_keys = [
                    key
                    for key in required_prediction_keys
                    if key not in candidate_result
                ]

                if missing_keys:
                    raise ValueError(
                        "Response is missing keys: "
                        f"{missing_keys}"
                    )

                break

            except Exception as error:
                last_error = error

                print(
                    "Attempt failed:",
                    type(error).__name__,
                    str(error),
                )

                if attempt < MAX_REQUEST_ATTEMPTS:
                    print(
                        "Waiting before retry..."
                    )

                    time.sleep(
                        REQUEST_INTERVAL_SECONDS
                    )

        if candidate_result is None:
            raise last_error

        # ------------------------------------------
        # Add reproducibility metadata
        # ------------------------------------------

        candidate_result[
            "school_id"
        ] = school_id

        candidate_result[
            "school_name"
        ] = school_name

        candidate_result[
            "model_id"
        ] = GROQ_MODEL_ID

        candidate_result[
            "prompt_version"
        ] = PROMPT_VERSION

        candidate_result[
            "generated_at_utc"
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        candidate_result[
            "imagery_resolution_m"
        ] = float(
            resolution_x
        )

        candidate_result[
            "imagery_vintage"
        ] = imagery_vintage

        candidate_result[
            "geotiff_file"
        ] = str(
            geotiff_path
        )

        candidate_result[
            "prediction_status"
        ] = "success"

        print(
            "New prediction generated."
        )

    except Exception as error:
        print(
            "Prediction failed:",
            type(error).__name__,
            str(error),
        )

        candidate_result = {
            "school_id": school_id,
            "school_name": school_name,
            "model_id": GROQ_MODEL_ID,
            "prompt_version": PROMPT_VERSION,
            "generated_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "imagery_resolution_m": (
                float(resolution_x)
                if resolution_x is not None
                else None
            ),
            "imagery_vintage": imagery_vintage,
            "prediction_status": "failed",
            "error_type": type(error).__name__,
            "error_message": str(error),
        }

    # --------------------------------------------------
    # Overwrite this school's existing JSON
    # --------------------------------------------------

    with json_output_path.open(
        "w",
        encoding="utf-8",
    ) as file_handle:
        json.dump(
            candidate_result,
            file_handle,
            indent=2,
        )

    batch_results.append(
        candidate_result
    )

    print(
        "JSON entry overwritten:",
        json_output_path.name,
    )

    # --------------------------------------------------
    # Overwrite CSV checkpoint after every school
    # --------------------------------------------------

    predictions_table = (
        make_predictions_table(
            batch_results
        )
    )

    predictions_table.to_csv(
        QWEN_PREDICTIONS_PATH,
        index=False,
    )

    print(
        "CSV checkpoint updated."
    )

    # --------------------------------------------------
    # Display this school's prediction
    # --------------------------------------------------

    display_result = (
        candidate_result.copy()
    )

    for json_field in [
        "uncertain_fields",
        "evidence_notes",
    ]:
        if json_field in display_result:
            display_result[
                json_field
            ] = json.dumps(
                display_result[
                    json_field
                ]
            )

    display(
        pd.DataFrame(
            [display_result]
        ).T.rename(
            columns={
                0: "Qwen candidate",
            }
        )
    )

    # --------------------------------------------------
    # Display original image after this school
    # --------------------------------------------------

    if image is not None:
        plt.figure(
            figsize=(15, 15)
        )

        plt.imshow(
            image
        )

        title = (
            f"{school_name}\n"
            f"NAIP aerial imagery"
        )

        if imagery_vintage is not None:
            title += (
                f" — {imagery_vintage}"
            )

        if resolution_x is not None:
            title += (
                f" — {resolution_x:.2f} m/pixel"
            )

        plt.title(
            title
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()

    else:
        print(
            "The aerial image could not be displayed "
            "because loading it failed."
        )

    # --------------------------------------------------
    # Wait before processing the next school
    # --------------------------------------------------

    if position < len(
        master_metadata
    ):
        print(
            f"\nWaiting "
            f"{REQUEST_INTERVAL_SECONDS} seconds "
            "before the next school..."
        )

        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )


# --------------------------------------------------
# Final summary
# --------------------------------------------------

predictions_table = pd.read_csv(
    QWEN_PREDICTIONS_PATH,
    dtype={
        "school_id": "string",
    },
)

successful_predictions = (
    predictions_table[
        "prediction_status"
    ]
    .eq("success")
    .sum()
)

failed_predictions = (
    predictions_table[
        "prediction_status"
    ]
    .eq("failed")
    .sum()
)

print("\n" + "=" * 70)
print("QWEN RAW-PREDICTION SUMMARY")
print("=" * 70)

print(
    "Total schools processed:",
    len(predictions_table),
)

print(
    "Successful predictions:",
    successful_predictions,
)

print(
    "Failed predictions:",
    failed_predictions,
)

print(
    "Predictions CSV:",
    QWEN_PREDICTIONS_PATH,
)


summary_columns = [
    "school_id",
    "school_name",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "human_review_required",
    "prediction_status",
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in predictions_table.columns
]

display(
    predictions_table[
        available_summary_columns
    ]
)

In [ ]:
# ==================================================
# SELF-CONTAINED CELL:
# Retry only failed Qwen predictions
# ==================================================

from datetime import datetime, timezone
from google.colab import drive, userdata
from groq import Groq
from io import BytesIO
from pathlib import Path
from PIL import Image
from IPython.display import display

import base64
import json
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio


# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------

drive.mount(
    "/content/drive"
)


# --------------------------------------------------
# Configure paths
# --------------------------------------------------

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "school_imagery_assignment"
)

MASTER_METADATA_PATH = (
    PROJECT_DIRECTORY
    / "naip_master_1200m"
    / "naip_master_metadata.csv"
)

QWEN_OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "qwen_predictions"
)

QWEN_JSON_DIRECTORY = (
    QWEN_OUTPUT_DIRECTORY
    / "raw_json"
)

QWEN_PREDICTIONS_PATH = (
    QWEN_OUTPUT_DIRECTORY
    / "qwen_predictions_raw.csv"
)

QWEN_JSON_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Check required files
# --------------------------------------------------

if not MASTER_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file not found: "
        f"{MASTER_METADATA_PATH}"
    )

if not QWEN_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "The existing Qwen predictions CSV "
        "was not found:\n"
        f"{QWEN_PREDICTIONS_PATH}"
    )


# --------------------------------------------------
# Load school metadata
# --------------------------------------------------

master_metadata = pd.read_csv(
    MASTER_METADATA_PATH,
    dtype={
        "school_id": "string",
    },
)

master_metadata[
    "school_id"
] = master_metadata[
    "school_id"
].astype("string")


# --------------------------------------------------
# Load existing predictions
# --------------------------------------------------

existing_predictions = pd.read_csv(
    QWEN_PREDICTIONS_PATH,
    dtype={
        "school_id": "string",
    },
)

existing_predictions[
    "school_id"
] = existing_predictions[
    "school_id"
].astype("string")

if "prediction_status" not in (
    existing_predictions.columns
):
    raise ValueError(
        "The predictions CSV does not contain "
        "'prediction_status'."
    )


# --------------------------------------------------
# Configure Groq
# --------------------------------------------------

GROQ_MODEL_ID = (
    "qwen/qwen3.6-27b"
)

PROMPT_VERSION = (
    "qwen_aerial_attributes_v2_rewrite_all"
)

REQUEST_INTERVAL_SECONDS = 65
MAX_REQUEST_ATTEMPTS = 2

api_key = userdata.get(
    "GROQ_API_KEY"
)

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY was not found in "
        "Colab Secrets."
    )

client = Groq(
    api_key=api_key
)


# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def safe_filename(value):
    safe_value = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value),
    )

    return safe_value.strip("_")


def load_geotiff_image(
    geotiff_path,
):
    with rasterio.open(
        geotiff_path
    ) as source:
        rgb_array = source.read(
            [1, 2, 3]
        )

        resolution_x = abs(
            source.transform.a
        )

        resolution_y = abs(
            source.transform.e
        )

    rgb_array = np.moveaxis(
        rgb_array,
        0,
        -1,
    )

    if rgb_array.dtype != np.uint8:
        valid_pixels = rgb_array[
            np.any(
                rgb_array != 0,
                axis=2,
            )
        ]

        if not valid_pixels.size:
            raise ValueError(
                "GeoTIFF contains no valid "
                "RGB pixels."
            )

        lower, upper = np.percentile(
            valid_pixels,
            [2, 98],
        )

        if upper <= lower:
            raise ValueError(
                "Unable to scale RGB values."
            )

        rgb_array = np.clip(
            (
                rgb_array.astype(
                    np.float32
                )
                - lower
            )
            / (upper - lower),
            0,
            1,
        )

        rgb_array = (
            rgb_array * 255
        ).astype(
            np.uint8
        )

    pil_image = Image.fromarray(
        rgb_array
    ).convert("RGB")

    return (
        pil_image,
        resolution_x,
        resolution_y,
    )


def image_to_data_url(
    pil_image,
    quality=85,
):
    buffer = BytesIO()

    pil_image.convert(
        "RGB"
    ).save(
        buffer,
        format="JPEG",
        quality=quality,
        optimize=True,
    )

    encoded_image = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return (
        "data:image/jpeg;base64,"
        + encoded_image
    )


def parse_json_response(
    response_text,
):
    try:
        return json.loads(
            response_text
        )

    except json.JSONDecodeError:
        cleaned_response = (
            response_text
            .strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )

        return json.loads(
            cleaned_response
        )


def prepare_csv_table(
    results,
):
    table = pd.DataFrame(
        results
    )

    for json_column in [
        "uncertain_fields",
        "evidence_notes",
    ]:
        if json_column in table.columns:
            table[
                json_column
            ] = table[
                json_column
            ].apply(
                lambda value: (
                    json.dumps(value)
                    if isinstance(
                        value,
                        (dict, list),
                    )
                    else value
                )
            )

    return table


# --------------------------------------------------
# Required Qwen fields
# --------------------------------------------------

required_prediction_keys = [
    "campus_visibility",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "uncertain_fields",
    "evidence_notes",
    "human_review_required",
]


# --------------------------------------------------
# Store all existing results
#
# Successful entries remain unchanged unless their
# school was previously marked failed.
# --------------------------------------------------

results_by_school_id = {
    str(row["school_id"]): row.to_dict()
    for _, row in (
        existing_predictions.iterrows()
    )
}


# --------------------------------------------------
# Select only failed schools
# --------------------------------------------------

failed_school_ids = set(
    existing_predictions.loc[
        existing_predictions[
            "prediction_status"
        ].eq("failed"),
        "school_id",
    ].astype(str)
)

retry_metadata = (
    master_metadata[
        master_metadata[
            "school_id"
        ]
        .astype(str)
        .isin(failed_school_ids)
    ]
    .copy()
    .reset_index(drop=True)
)

successful_before_retry = (
    existing_predictions[
        "prediction_status"
    ].eq("success").sum()
)

print(
    "Successful predictions preserved:",
    successful_before_retry,
)

print(
    "Failed schools to retry:",
    len(retry_metadata),
)

if retry_metadata.empty:
    print(
        "There are no failed schools to retry."
    )


# --------------------------------------------------
# Retry failed schools
# --------------------------------------------------

daily_limit_reached = False

for retry_index, school in (
    retry_metadata.iterrows()
):
    retry_number = retry_index + 1

    school_id = str(
        school["school_id"]
    )

    school_name = str(
        school["school_name"]
    )

    output_name = (
        f"{school_id}_"
        f"{safe_filename(school_name)}.json"
    )

    json_output_path = (
        QWEN_JSON_DIRECTORY
        / output_name
    )

    image = None
    resolution_x = None
    resolution_y = None
    imagery_vintage = None
    candidate_result = None
    last_error = None

    print("\n" + "=" * 70)

    print(
        f"Retry [{retry_number:02d}/"
        f"{len(retry_metadata)}] "
        f"{school_name}"
    )

    print("=" * 70)

    try:
        # ------------------------------------------
        # Load aerial imagery
        # ------------------------------------------

        geotiff_path = Path(
            school["geotiff_file"]
        )

        if not geotiff_path.exists():
            raise FileNotFoundError(
                f"GeoTIFF not found: "
                f"{geotiff_path}"
            )

        (
            image,
            resolution_x,
            resolution_y,
        ) = load_geotiff_image(
            geotiff_path
        )

        if (
            "naip_year" in school.index
            and pd.notna(
                school["naip_year"]
            )
        ):
            imagery_vintage = int(
                school["naip_year"]
            )

        # ------------------------------------------
        # Create overview and two zoom strips
        # ------------------------------------------

        strip_height = 1200

        upper_strip = image.crop(
            (
                0,
                0,
                image.width,
                min(
                    strip_height,
                    image.height,
                ),
            )
        )

        lower_start = max(
            0,
            image.height - strip_height,
        )

        lower_strip = image.crop(
            (
                0,
                lower_start,
                image.width,
                image.height,
            )
        )

        overview_data_url = (
            image_to_data_url(
                image
            )
        )

        upper_data_url = (
            image_to_data_url(
                upper_strip
            )
        )

        lower_data_url = (
            image_to_data_url(
                lower_strip
            )
        )

        # ------------------------------------------
        # Qwen prompt
        # ------------------------------------------

        prompt = f"""
You are reviewing NAIP aerial imagery of:

School: {school_name}

The imagery resolution is approximately
{resolution_x} metres per pixel.

Image 1 is the complete 1,200 metre school-area overview.
Image 2 is an enlarged upper portion of Image 1.
Image 3 is an enlarged lower portion of Image 1.

The reported school coordinate may be imperfect. Identify
the likely school campus carefully. If campus identity or
extent is unclear, mark the relevant fields uncertain.

Return exactly one valid JSON object:

{{
  "campus_visibility": "clear, partial, or poor",
  "rooftop_solar_present": "yes, no, or uncertain",
  "rooftop_solar_area_m2_estimate": number or null,
  "portable_classroom_count": integer or null,
  "pool_present": "yes, no, or uncertain",
  "running_track": "yes, no, or uncertain",
  "full_size_sports_fields_count": integer or null,
  "hard_courts_count": integer or null,
  "uncertain_fields": ["field names requiring review"],
  "evidence_notes": {{
    "solar": "brief evidence and approximate location",
    "portable_classrooms": "brief evidence and location",
    "pool": "brief evidence and location",
    "running_track": "brief evidence and location",
    "sports_fields": "brief evidence and location",
    "hard_courts": "brief evidence and location"
  }},
  "human_review_required": true or false
}}

Measurement definitions:

1. Rooftop solar presence:
   Report yes only for recognizable rooftop photovoltaic
   panel arrays. Do not confuse HVAC equipment, skylights,
   roof shadows or dark roofing with solar panels.

2. Rooftop solar area:
   If solar is present, estimate the total visible
   panel-covered area in square metres. This is panel area,
   not total roof area. Otherwise return null.

3. Portable classrooms:
   Count distinct portable or modular classroom buildings.
   Do not count individual rooms. Do not count permanent
   buildings, houses, storage sheds or ordinary trailers.

4. Pool:
   Report yes only for a recognizable swimming pool.
   Do not confuse ponds, blue roofs, artificial turf or
   shadows with pools.

5. Running track:
   Report yes for a recognizable closed oval running track,
   commonly surrounding a rectangular sports field.

6. Full-size sports fields:
   Count recognizable full-size fields intended for sports.
   A field inside a running track counts as one field.
   Do not count small lawns or generic unmarked open spaces.

7. Hard courts:
   Count distinct individual playable outdoor courts with
   recognizable sport markings. Parking lots, parking-space
   markings and paved school yards without court markings
   are not hard courts.

Additional rules:

- Measure only facilities belonging to the target campus.
- Do not generate bounding boxes.
- Use approximate location descriptions such as
  bottom-centre, upper-left or beside the main building.
- Use null or uncertain rather than guessing.
- Do not generate confidence scores.
- These are raw model predictions for later validation
  and human review.
"""

        request_content = [
            {
                "type": "text",
                "text": prompt,
            },
            {
                "type": "text",
                "text": (
                    "Image 1: complete overview"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": overview_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 2: enlarged upper portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": upper_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 3: enlarged lower portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": lower_data_url,
                },
            },
        ]

        # ------------------------------------------
        # Call Groq
        # ------------------------------------------

        for attempt in range(
            1,
            MAX_REQUEST_ATTEMPTS + 1,
        ):
            try:
                print(
                    f"API attempt "
                    f"{attempt}/"
                    f"{MAX_REQUEST_ATTEMPTS}"
                )

                completion = (
                    client
                    .chat
                    .completions
                    .create(
                        model=GROQ_MODEL_ID,
                        messages=[
                            {
                                "role": "user",
                                "content": (
                                    request_content
                                ),
                            }
                        ],
                        temperature=0.5,
                        reasoning_effort="none",
                        max_completion_tokens=800,
                        response_format={
                            "type": "json_object",
                        },
                    )
                )

                response_text = (
                    completion
                    .choices[0]
                    .message
                    .content
                )

                if not response_text:
                    raise RuntimeError(
                        "Groq returned an "
                        "empty response."
                    )

                candidate_result = (
                    parse_json_response(
                        response_text
                    )
                )

                missing_keys = [
                    key
                    for key in required_prediction_keys
                    if key not in candidate_result
                ]

                if missing_keys:
                    raise ValueError(
                        "Response is missing keys: "
                        f"{missing_keys}"
                    )

                break

            except Exception as error:
                last_error = error
                error_text = str(
                    error
                ).lower()

                print(
                    "Attempt failed:",
                    type(error).__name__,
                    str(error),
                )

                # Waiting 65 seconds will not fix
                # a daily token quota.
                if (
                    "tokens per day" in error_text
                    or "tpd" in error_text
                ):
                    daily_limit_reached = True

                    print(
                        "\nGroq's daily token limit "
                        "is still active."
                    )

                    print(
                        "Stopping without overwriting "
                        "existing predictions."
                    )

                    break

                if attempt < MAX_REQUEST_ATTEMPTS:
                    print(
                        "Waiting before retry..."
                    )

                    time.sleep(
                        REQUEST_INTERVAL_SECONDS
                    )

        if candidate_result is None:
            if last_error is not None:
                raise last_error

            raise RuntimeError(
                "No prediction was generated."
            )

        # ------------------------------------------
        # Add reproducibility metadata
        # ------------------------------------------

        candidate_result[
            "school_id"
        ] = school_id

        candidate_result[
            "school_name"
        ] = school_name

        candidate_result[
            "model_id"
        ] = GROQ_MODEL_ID

        candidate_result[
            "prompt_version"
        ] = PROMPT_VERSION

        candidate_result[
            "generated_at_utc"
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        candidate_result[
            "imagery_resolution_m"
        ] = float(
            resolution_x
        )

        candidate_result[
            "imagery_vintage"
        ] = imagery_vintage

        candidate_result[
            "geotiff_file"
        ] = str(
            geotiff_path
        )

        candidate_result[
            "prediction_status"
        ] = "success"

        # ------------------------------------------
        # Replace only the failed school
        # ------------------------------------------

        results_by_school_id[
            school_id
        ] = candidate_result

        with json_output_path.open(
            "w",
            encoding="utf-8",
        ) as file_handle:
            json.dump(
                candidate_result,
                file_handle,
                indent=2,
            )

        # ------------------------------------------
        # Rebuild CSV in original school order
        # ------------------------------------------

        ordered_results = []

        for _, metadata_row in (
            master_metadata.iterrows()
        ):
            metadata_school_id = str(
                metadata_row["school_id"]
            )

            if metadata_school_id in (
                results_by_school_id
            ):
                ordered_results.append(
                    results_by_school_id[
                        metadata_school_id
                    ]
                )

        predictions_table = (
            prepare_csv_table(
                ordered_results
            )
        )

        predictions_table.to_csv(
            QWEN_PREDICTIONS_PATH,
            index=False,
        )

        print(
            "Prediction succeeded."
        )

        print(
            "JSON and CSV entries updated."
        )

        # ------------------------------------------
        # Display prediction
        # ------------------------------------------

        display_result = (
            candidate_result.copy()
        )

        for json_field in [
            "uncertain_fields",
            "evidence_notes",
        ]:
            if json_field in display_result:
                display_result[
                    json_field
                ] = json.dumps(
                    display_result[
                        json_field
                    ]
                )

        display(
            pd.DataFrame(
                [display_result]
            ).T.rename(
                columns={
                    0: "Qwen candidate",
                }
            )
        )

        # ------------------------------------------
        # Display original aerial image
        # ------------------------------------------

        plt.figure(
            figsize=(15, 15)
        )

        plt.imshow(
            image
        )

        title = (
            f"{school_name}\n"
            "NAIP aerial imagery"
        )

        if imagery_vintage is not None:
            title += (
                f" — {imagery_vintage}"
            )

        title += (
            f" — {resolution_x:.2f} m/pixel"
        )

        plt.title(
            title
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()

    except Exception as error:
        print(
            "School remains failed:",
            type(error).__name__,
            str(error),
        )

    # If TPD is still exhausted, stop after one call.
    if daily_limit_reached:
        break

    # Space requests for Groq's other limits.
    if retry_number < len(
        retry_metadata
    ):
        print(
            f"\nWaiting "
            f"{REQUEST_INTERVAL_SECONDS} seconds "
            "before the next school..."
        )

        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )


# --------------------------------------------------
# Final summary
# --------------------------------------------------

updated_predictions = pd.read_csv(
    QWEN_PREDICTIONS_PATH,
    dtype={
        "school_id": "string",
    },
)

successful_count = (
    updated_predictions[
        "prediction_status"
    ].eq("success").sum()
)

failed_count = (
    updated_predictions[
        "prediction_status"
    ].eq("failed").sum()
)

print("\n" + "=" * 70)
print("UPDATED QWEN PREDICTION SUMMARY")
print("=" * 70)

print(
    "Total schools:",
    len(updated_predictions),
)

print(
    "Successful predictions:",
    successful_count,
)

print(
    "Failed predictions remaining:",
    failed_count,
)

print(
    "Predictions CSV:",
    QWEN_PREDICTIONS_PATH,
)

summary_columns = [
    "school_id",
    "school_name",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "human_review_required",
    "prediction_status",
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in updated_predictions.columns
]

display(
    updated_predictions[
        available_summary_columns
    ]
)

In [ ]:
# ==================================================
# SELF-CONTAINED CELL:
# Retry only failed Qwen predictions
# ==================================================

from datetime import datetime, timezone
from google.colab import drive, userdata
from groq import Groq
from io import BytesIO
from pathlib import Path
from PIL import Image
from IPython.display import display

import base64
import json
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio


# --------------------------------------------------
# Mount Google Drive
# --------------------------------------------------

drive.mount(
    "/content/drive"
)


# --------------------------------------------------
# Configure paths
# --------------------------------------------------

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/"
    "school_imagery_assignment"
)

MASTER_METADATA_PATH = (
    PROJECT_DIRECTORY
    / "naip_master_1200m"
    / "naip_master_metadata.csv"
)

QWEN_OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY
    / "qwen_predictions"
)

QWEN_JSON_DIRECTORY = (
    QWEN_OUTPUT_DIRECTORY
    / "raw_json"
)

QWEN_PREDICTIONS_PATH = (
    QWEN_OUTPUT_DIRECTORY
    / "qwen_predictions_raw.csv"
)

QWEN_JSON_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------
# Check required files
# --------------------------------------------------

if not MASTER_METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file not found: "
        f"{MASTER_METADATA_PATH}"
    )

if not QWEN_PREDICTIONS_PATH.exists():
    raise FileNotFoundError(
        "The existing Qwen predictions CSV "
        "was not found:\n"
        f"{QWEN_PREDICTIONS_PATH}"
    )


# --------------------------------------------------
# Load school metadata
# --------------------------------------------------

master_metadata = pd.read_csv(
    MASTER_METADATA_PATH,
    dtype={
        "school_id": "string",
    },
)

master_metadata[
    "school_id"
] = master_metadata[
    "school_id"
].astype("string")


# --------------------------------------------------
# Load existing predictions
# --------------------------------------------------

existing_predictions = pd.read_csv(
    QWEN_PREDICTIONS_PATH,
    dtype={
        "school_id": "string",
    },
)

existing_predictions[
    "school_id"
] = existing_predictions[
    "school_id"
].astype("string")

if "prediction_status" not in (
    existing_predictions.columns
):
    raise ValueError(
        "The predictions CSV does not contain "
        "'prediction_status'."
    )


# --------------------------------------------------
# Configure Groq
# --------------------------------------------------

GROQ_MODEL_ID = (
    "qwen/qwen3.6-27b"
)

PROMPT_VERSION = (
    "qwen_aerial_attributes_v2_rewrite_all"
)

REQUEST_INTERVAL_SECONDS = 65
MAX_REQUEST_ATTEMPTS = 2

api_key = userdata.get(
    "GROQ_API_KEY"
)

if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY was not found in "
        "Colab Secrets."
    )

client = Groq(
    api_key=api_key
)


# --------------------------------------------------
# Helper functions
# --------------------------------------------------

def safe_filename(value):
    safe_value = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        str(value),
    )

    return safe_value.strip("_")


def load_geotiff_image(
    geotiff_path,
):
    with rasterio.open(
        geotiff_path
    ) as source:
        rgb_array = source.read(
            [1, 2, 3]
        )

        resolution_x = abs(
            source.transform.a
        )

        resolution_y = abs(
            source.transform.e
        )

    rgb_array = np.moveaxis(
        rgb_array,
        0,
        -1,
    )

    if rgb_array.dtype != np.uint8:
        valid_pixels = rgb_array[
            np.any(
                rgb_array != 0,
                axis=2,
            )
        ]

        if not valid_pixels.size:
            raise ValueError(
                "GeoTIFF contains no valid "
                "RGB pixels."
            )

        lower, upper = np.percentile(
            valid_pixels,
            [2, 98],
        )

        if upper <= lower:
            raise ValueError(
                "Unable to scale RGB values."
            )

        rgb_array = np.clip(
            (
                rgb_array.astype(
                    np.float32
                )
                - lower
            )
            / (upper - lower),
            0,
            1,
        )

        rgb_array = (
            rgb_array * 255
        ).astype(
            np.uint8
        )

    pil_image = Image.fromarray(
        rgb_array
    ).convert("RGB")

    return (
        pil_image,
        resolution_x,
        resolution_y,
    )


def image_to_data_url(
    pil_image,
    quality=85,
):
    buffer = BytesIO()

    pil_image.convert(
        "RGB"
    ).save(
        buffer,
        format="JPEG",
        quality=quality,
        optimize=True,
    )

    encoded_image = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    return (
        "data:image/jpeg;base64,"
        + encoded_image
    )


def parse_json_response(
    response_text,
):
    try:
        return json.loads(
            response_text
        )

    except json.JSONDecodeError:
        cleaned_response = (
            response_text
            .strip()
            .removeprefix("```json")
            .removeprefix("```")
            .removesuffix("```")
            .strip()
        )

        return json.loads(
            cleaned_response
        )


def prepare_csv_table(
    results,
):
    table = pd.DataFrame(
        results
    )

    for json_column in [
        "uncertain_fields",
        "evidence_notes",
    ]:
        if json_column in table.columns:
            table[
                json_column
            ] = table[
                json_column
            ].apply(
                lambda value: (
                    json.dumps(value)
                    if isinstance(
                        value,
                        (dict, list),
                    )
                    else value
                )
            )

    return table


# --------------------------------------------------
# Required Qwen fields
# --------------------------------------------------

required_prediction_keys = [
    "campus_visibility",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "uncertain_fields",
    "evidence_notes",
    "human_review_required",
]


# --------------------------------------------------
# Store all existing results
#
# Successful entries remain unchanged unless their
# school was previously marked failed.
# --------------------------------------------------

results_by_school_id = {
    str(row["school_id"]): row.to_dict()
    for _, row in (
        existing_predictions.iterrows()
    )
}


# --------------------------------------------------
# Select only failed schools
# --------------------------------------------------

RETRY_SCHOOL_IDS = {
    "130333001308",
    "450387001491",
    "481455000943",
    "481623001325",
    "550957000382",
}

retry_metadata = (
    master_metadata[
        master_metadata[
            "school_id"
        ]
        .astype(str)
        .isin(RETRY_SCHOOL_IDS)
    ]
    .copy()
    .reset_index(drop=True)
)

successful_before_retry = (
    existing_predictions[
        "prediction_status"
    ].eq("success").sum()
)

print(
    "Successful predictions preserved:",
    successful_before_retry,
)

print(
    "Schools selected for retry:",
    len(retry_metadata),
)

print(
    "School IDs being retried:",
    retry_metadata[
        "school_id"
    ].astype(str).tolist(),
)

if retry_metadata.empty:
    print(
        "None of the requested school IDs "
        "were found in the metadata."
    )


# --------------------------------------------------
# Retry failed schools
# --------------------------------------------------

daily_limit_reached = False

for retry_index, school in (
    retry_metadata.iterrows()
):
    retry_number = retry_index + 1

    school_id = str(
        school["school_id"]
    )

    school_name = str(
        school["school_name"]
    )

    output_name = (
        f"{school_id}_"
        f"{safe_filename(school_name)}.json"
    )

    json_output_path = (
        QWEN_JSON_DIRECTORY
        / output_name
    )

    image = None
    resolution_x = None
    resolution_y = None
    imagery_vintage = None
    candidate_result = None
    last_error = None

    print("\n" + "=" * 70)

    print(
        f"Retry [{retry_number:02d}/"
        f"{len(retry_metadata)}] "
        f"{school_name}"
    )

    print("=" * 70)

    try:
        # ------------------------------------------
        # Load aerial imagery
        # ------------------------------------------

        geotiff_path = Path(
            school["geotiff_file"]
        )

        if not geotiff_path.exists():
            raise FileNotFoundError(
                f"GeoTIFF not found: "
                f"{geotiff_path}"
            )

        (
            image,
            resolution_x,
            resolution_y,
        ) = load_geotiff_image(
            geotiff_path
        )

        if (
            "naip_year" in school.index
            and pd.notna(
                school["naip_year"]
            )
        ):
            imagery_vintage = int(
                school["naip_year"]
            )

        # ------------------------------------------
        # Create overview and two zoom strips
        # ------------------------------------------

        strip_height = 1200

        upper_strip = image.crop(
            (
                0,
                0,
                image.width,
                min(
                    strip_height,
                    image.height,
                ),
            )
        )

        lower_start = max(
            0,
            image.height - strip_height,
        )

        lower_strip = image.crop(
            (
                0,
                lower_start,
                image.width,
                image.height,
            )
        )

        overview_data_url = (
            image_to_data_url(
                image
            )
        )

        upper_data_url = (
            image_to_data_url(
                upper_strip
            )
        )

        lower_data_url = (
            image_to_data_url(
                lower_strip
            )
        )

        # ------------------------------------------
        # Qwen prompt
        # ------------------------------------------

        prompt = f"""
You are reviewing NAIP aerial imagery of:

School: {school_name}

The imagery resolution is approximately
{resolution_x} metres per pixel.

Image 1 is the complete 1,200 metre school-area overview.
Image 2 is an enlarged upper portion of Image 1.
Image 3 is an enlarged lower portion of Image 1.

The reported school coordinate may be imperfect. Identify
the likely school campus carefully. If campus identity or
extent is unclear, mark the relevant fields uncertain.

Return exactly one valid JSON object:

{{
  "campus_visibility": "clear, partial, or poor",
  "rooftop_solar_present": "yes, no, or uncertain",
  "rooftop_solar_area_m2_estimate": number or null,
  "portable_classroom_count": integer or null,
  "pool_present": "yes, no, or uncertain",
  "running_track": "yes, no, or uncertain",
  "full_size_sports_fields_count": integer or null,
  "hard_courts_count": integer or null,
  "uncertain_fields": ["field names requiring review"],
  "evidence_notes": {{
    "solar": "brief evidence and approximate location",
    "portable_classrooms": "brief evidence and location",
    "pool": "brief evidence and location",
    "running_track": "brief evidence and location",
    "sports_fields": "brief evidence and location",
    "hard_courts": "brief evidence and location"
  }},
  "human_review_required": true or false
}}

Measurement definitions:

1. Rooftop solar presence:
   Report yes only for recognizable rooftop photovoltaic
   panel arrays. Do not confuse HVAC equipment, skylights,
   roof shadows or dark roofing with solar panels.

2. Rooftop solar area:
   If solar is present, estimate the total visible
   panel-covered area in square metres. This is panel area,
   not total roof area. Otherwise return null.

3. Portable classrooms:
   Count distinct portable or modular classroom buildings.
   Do not count individual rooms. Do not count permanent
   buildings, houses, storage sheds or ordinary trailers.

4. Pool:
   Report yes only for a recognizable swimming pool.
   Do not confuse ponds, blue roofs, artificial turf or
   shadows with pools.

5. Running track:
   Report yes for a recognizable closed oval running track,
   commonly surrounding a rectangular sports field.

6. Full-size sports fields:
   Count recognizable full-size fields intended for sports.
   A field inside a running track counts as one field.
   Do not count small lawns or generic unmarked open spaces.

7. Hard courts:
   Count distinct individual playable outdoor courts with
   recognizable sport markings. Parking lots, parking-space
   markings and paved school yards without court markings
   are not hard courts.

Additional rules:

- Measure only facilities belonging to the target campus.
- Do not generate bounding boxes.
- Use approximate location descriptions such as
  bottom-centre, upper-left or beside the main building.
- Use null or uncertain rather than guessing.
- Do not generate confidence scores.
- These are raw model predictions for later validation
  and human review.
"""

        request_content = [
            {
                "type": "text",
                "text": prompt,
            },
            {
                "type": "text",
                "text": (
                    "Image 1: complete overview"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": overview_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 2: enlarged upper portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": upper_data_url,
                },
            },
            {
                "type": "text",
                "text": (
                    "Image 3: enlarged lower portion"
                ),
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": lower_data_url,
                },
            },
        ]

        # ------------------------------------------
        # Call Groq
        # ------------------------------------------

        for attempt in range(
            1,
            MAX_REQUEST_ATTEMPTS + 1,
        ):
            try:
                print(
                    f"API attempt "
                    f"{attempt}/"
                    f"{MAX_REQUEST_ATTEMPTS}"
                )

                completion = (
                    client
                    .chat
                    .completions
                    .create(
                        model=GROQ_MODEL_ID,
                        messages=[
                            {
                                "role": "user",
                                "content": (
                                    request_content
                                ),
                            }
                        ],
                        temperature=0.5,
                        reasoning_effort="none",
                        max_completion_tokens=800,
                        response_format={
                            "type": "json_object",
                        },
                    )
                )

                response_text = (
                    completion
                    .choices[0]
                    .message
                    .content
                )

                if not response_text:
                    raise RuntimeError(
                        "Groq returned an "
                        "empty response."
                    )

                candidate_result = (
                    parse_json_response(
                        response_text
                    )
                )

                missing_keys = [
                    key
                    for key in required_prediction_keys
                    if key not in candidate_result
                ]

                if missing_keys:
                    raise ValueError(
                        "Response is missing keys: "
                        f"{missing_keys}"
                    )

                break

            except Exception as error:
                last_error = error
                error_text = str(
                    error
                ).lower()

                print(
                    "Attempt failed:",
                    type(error).__name__,
                    str(error),
                )

                # Waiting 65 seconds will not fix
                # a daily token quota.
                if (
                    "tokens per day" in error_text
                    or "tpd" in error_text
                ):
                    daily_limit_reached = True

                    print(
                        "\nGroq's daily token limit "
                        "is still active."
                    )

                    print(
                        "Stopping without overwriting "
                        "existing predictions."
                    )

                    break

                if attempt < MAX_REQUEST_ATTEMPTS:
                    print(
                        "Waiting before retry..."
                    )

                    time.sleep(
                        REQUEST_INTERVAL_SECONDS
                    )

        if candidate_result is None:
            if last_error is not None:
                raise last_error

            raise RuntimeError(
                "No prediction was generated."
            )

        # ------------------------------------------
        # Add reproducibility metadata
        # ------------------------------------------

        candidate_result[
            "school_id"
        ] = school_id

        candidate_result[
            "school_name"
        ] = school_name

        candidate_result[
            "model_id"
        ] = GROQ_MODEL_ID

        candidate_result[
            "prompt_version"
        ] = PROMPT_VERSION

        candidate_result[
            "generated_at_utc"
        ] = datetime.now(
            timezone.utc
        ).isoformat()

        candidate_result[
            "imagery_resolution_m"
        ] = float(
            resolution_x
        )

        candidate_result[
            "imagery_vintage"
        ] = imagery_vintage

        candidate_result[
            "geotiff_file"
        ] = str(
            geotiff_path
        )

        candidate_result[
            "prediction_status"
        ] = "success"

        # ------------------------------------------
        # Replace only the failed school
        # ------------------------------------------

        results_by_school_id[
            school_id
        ] = candidate_result

        with json_output_path.open(
            "w",
            encoding="utf-8",
        ) as file_handle:
            json.dump(
                candidate_result,
                file_handle,
                indent=2,
            )

        # ------------------------------------------
        # Rebuild CSV in original school order
        # ------------------------------------------

        ordered_results = []

        for _, metadata_row in (
            master_metadata.iterrows()
        ):
            metadata_school_id = str(
                metadata_row["school_id"]
            )

            if metadata_school_id in (
                results_by_school_id
            ):
                ordered_results.append(
                    results_by_school_id[
                        metadata_school_id
                    ]
                )

        predictions_table = (
            prepare_csv_table(
                ordered_results
            )
        )

        predictions_table.to_csv(
            QWEN_PREDICTIONS_PATH,
            index=False,
        )

        print(
            "Prediction succeeded."
        )

        print(
            "JSON and CSV entries updated."
        )

        # ------------------------------------------
        # Display prediction
        # ------------------------------------------

        display_result = (
            candidate_result.copy()
        )

        for json_field in [
            "uncertain_fields",
            "evidence_notes",
        ]:
            if json_field in display_result:
                display_result[
                    json_field
                ] = json.dumps(
                    display_result[
                        json_field
                    ]
                )

        display(
            pd.DataFrame(
                [display_result]
            ).T.rename(
                columns={
                    0: "Qwen candidate",
                }
            )
        )

        # ------------------------------------------
        # Display original aerial image
        # ------------------------------------------

        plt.figure(
            figsize=(15, 15)
        )

        plt.imshow(
            image
        )

        title = (
            f"{school_name}\n"
            "NAIP aerial imagery"
        )

        if imagery_vintage is not None:
            title += (
                f" — {imagery_vintage}"
            )

        title += (
            f" — {resolution_x:.2f} m/pixel"
        )

        plt.title(
            title
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()

    except Exception as error:
        print(
            "School remains failed:",
            type(error).__name__,
            str(error),
        )

    # If TPD is still exhausted, stop after one call.
    if daily_limit_reached:
        break

    # Space requests for Groq's other limits.
    if retry_number < len(
        retry_metadata
    ):
        print(
            f"\nWaiting "
            f"{REQUEST_INTERVAL_SECONDS} seconds "
            "before the next school..."
        )

        time.sleep(
            REQUEST_INTERVAL_SECONDS
        )


# --------------------------------------------------
# Final summary
# --------------------------------------------------

updated_predictions = pd.read_csv(
    QWEN_PREDICTIONS_PATH,
    dtype={
        "school_id": "string",
    },
)

successful_count = (
    updated_predictions[
        "prediction_status"
    ].eq("success").sum()
)

failed_count = (
    updated_predictions[
        "prediction_status"
    ].eq("failed").sum()
)

print("\n" + "=" * 70)
print("UPDATED QWEN PREDICTION SUMMARY")
print("=" * 70)

print(
    "Total schools:",
    len(updated_predictions),
)

print(
    "Successful predictions:",
    successful_count,
)

print(
    "Failed predictions remaining:",
    failed_count,
)

print(
    "Predictions CSV:",
    QWEN_PREDICTIONS_PATH,
)

summary_columns = [
    "school_id",
    "school_name",
    "rooftop_solar_present",
    "rooftop_solar_area_m2_estimate",
    "portable_classroom_count",
    "pool_present",
    "running_track",
    "full_size_sports_fields_count",
    "hard_courts_count",
    "human_review_required",
    "prediction_status",
]

available_summary_columns = [
    column
    for column in summary_columns
    if column in updated_predictions.columns
]

display(
    updated_predictions[
        available_summary_columns
    ]
)